<a href="https://colab.research.google.com/github/AlisaLyskova/medical_diseases_names_classification/blob/main/medical_diseases_names_classification_bert_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Text classification of ICD‑11 Russian disease titles with TensorFlow + BERT tokenizer.

- Loads ICD‑11 tabulation file (Russian version).
- Cleans and normalizes disease titles.
- Builds train / validation splits.
- Encodes labels for ICD blocks / groupings.
- Tokenizes text with BERT tokenizer.
- Trains a Bidirectional LSTM classifier on tokenized sequences.

In [1]:
from google.colab import files

uploaded = files.upload()

Saving SimpleTabulation-ICD-11-MMS-ru.txt to SimpleTabulation-ICD-11-MMS-ru.txt


In [18]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Bidirectional, Dropout, Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer  # pip install transformers


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Path to the ICD‑11 tabulation file (TSV)
DATA_PATH = "SimpleTabulation-ICD-11-MMS-ru.txt"

# Text and label columns
TITLE_COLUMN = "Title"
CODE_COLUMN = "NEW_ID"

# Tokenization / model hyperparameters
MAX_SEQ_LENGTH = 95
EMBEDDING_DIM = 128
LSTM_UNITS = 128
DENSE_UNITS = 64
DROPOUT_RATE = 0.5
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
EPOCHS = 30
VALIDATION_FRACTION = 0.2
RANDOM_STATE = 42


# ---------------------------------------------------------------------------
# Data loading and basic cleaning
# ---------------------------------------------------------------------------

def load_icd_data(path: str) -> pd.DataFrame:
    """
    Load ICD‑11 Russian tabulation file (TSV) into a pandas DataFrame.
    Keeps all columns as strings where possible to avoid dtype warnings.
    """
    df = pd.read_csv(path, sep="\t", low_memory=False)
    df = df.dropna(subset=['Title'])
    return df


def clean_icd_title(raw_title: str) -> str:
    """
    Clean raw ICD title strings.

    Typical cleaning:
    - Remove leading ' - ' and similar hyphen-based indentation markers
      used to indicate hierarchy levels.
    - Strip surrounding whitespace.
    """
    if not isinstance(raw_title, str):
        return ""

    # Remove sequences like "- - - " at the beginning of the line
    title = re.sub(r"^(-\s*)+", "", raw_title)
    # Normalize spaces
    title = re.sub(r"\s+", " ", title)
    return title.strip()


def add_clean_title_column(df: pd.DataFrame,
                           source_col: str = TITLE_COLUMN,
                           target_col: str = "TitleNEW") -> pd.DataFrame:
    """
    Add a column with cleaned disease titles.
    """
    df[target_col] = df[source_col].apply(clean_icd_title)
    return df

def new_group_id(df: pd.DataFrame)  -> pd.DataFrame:
    # Compute NEW_ID according to ICD-11 hierarchy logic
    # NEW_ID is:
    # - Code for first-level leaf diseases
    # - BlockId when first-level disease has no Code
    # - Code otherwise for first-level
    # - Grouping1 for all other (deeper) levels
    df["NEW_ID"] = np.where(
        ~(df["Title"].str.startswith("- -")) & df["Title"].str.startswith("- ") & (df["isLeaf"] == True),
        df["Code"],
        np.where(
            ~(df["Title"].str.startswith("- -")) & df["Title"].str.startswith("- ") & (df["Code"].isna()),
            df["BlockId"],
            np.where(
                ~(df["Title"].str.startswith("- -")) & df["Title"].str.startswith("- "),
                df["Code"],
                df["Grouping1"],
            ),
        ),
    )
    df = df.dropna(subset=['NEW_ID'])
    df = df.reset_index(drop=True)
    return df


# ---------------------------------------------------------------------------
# Label preparation
# ---------------------------------------------------------------------------

def prepare_labels(df: pd.DataFrame,
                   label_col: str = CODE_COLUMN):
    """
    Prepare labels for classification.

    - Takes a column with ICD block / grouping identifiers.
    - Drops rows without labels.
    - Encodes labels into integer IDs with LabelEncoder.
    - Returns:
        - filtered DataFrame (only rows with labels and text),
        - encoded label array,
        - LabelEncoder instance (for mapping ids <-> labels).
    """
    # Keep rows with non-empty labels
    mask = df[label_col].notna()
    df_filtered = df.loc[mask].copy()

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(df_filtered[label_col].astype(str))

    return df_filtered, y_encoded, label_encoder


# ---------------------------------------------------------------------------
# Train / validation split
# ---------------------------------------------------------------------------

def train_val_split(df: pd.DataFrame,
                    y: np.ndarray,
                    val_fraction: float = VALIDATION_FRACTION,
                    random_state: int = RANDOM_STATE):
    """
    Split data and labels into train / validation sets using random permutation.
    """
    n_samples = len(df)
    indices = np.random.RandomState(random_state).permutation(n_samples)

    n_val = int(n_samples * val_fraction)
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]

    df_train = df.iloc[train_idx]
    df_val = df.iloc[val_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]

    return df_train, df_val, y_train, y_val


# ---------------------------------------------------------------------------
# Text tokenization with BERT tokenizer
# ---------------------------------------------------------------------------

def process_terms(terms_list,
                  tokenizer: BertTokenizer,
                  max_length: int = MAX_SEQ_LENGTH):
    """
    Tokenize and vectorize a list of disease titles with a BERT tokenizer.

    - Adds special tokens ([CLS], [SEP]).
    - Truncates / pads to max_length.
    - Returns a 2D numpy array of token IDs suitable for Keras.
    """
    encoded_terms = [
        tokenizer.encode(
            term,
            add_special_tokens=True,
            max_length=max_length,
            padding="max_length",
            truncation=True
        )
        for term in terms_list
    ]

    padded_sequences = pad_sequences(
        encoded_terms,
        maxlen=max_length,
        padding="post",
        dtype="long"
    )

    return padded_sequences


# ---------------------------------------------------------------------------
# Model definition
# ---------------------------------------------------------------------------

def build_model(vocab_size: int,
                max_length: int,
                n_classes: int) -> tf.keras.Model:
    """
    Build a Bidirectional LSTM classification model on top of token IDs.

    NOTE:
    - This uses a simple Embedding + BiLSTM stack, not the full BERT transformer.
    - For better performance you can replace this with a transformer-based classifier.
    """
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=EMBEDDING_DIM,
            input_length=max_length
        ),
        Bidirectional(LSTM(units=LSTM_UNITS, return_sequences=False)),
        Dropout(DROPOUT_RATE),
        Dense(DENSE_UNITS, activation="relu"),
        Dense(n_classes, activation="softmax"),
    ])

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


# ---------------------------------------------------------------------------
# Main training pipeline
# ---------------------------------------------------------------------------

def main():
    # 1. Load data
    df = load_icd_data(DATA_PATH)
    print(f"Loaded {len(df)} rows from {DATA_PATH}")

    # 2. Add cleaned titles
    df = add_clean_title_column(df, source_col=TITLE_COLUMN, target_col="TitleNEW")

    # 3. Compute NEW_ID according to ICD-11 hierarchy logic
    df = new_group_id(df)

    # 4. Prepare labels
    df_labeled, y_encoded, label_encoder = prepare_labels(df, label_col=CODE_COLUMN)
    print(f"Number of labeled examples: {len(df_labeled)}")
    print(f"Number of classes: {len(label_encoder.classes_)}")

    # 5. Train / validation split
    df_train, df_val, y_train, y_val = train_val_split(df_labeled, y_encoded)

    diseases_for_train = df_train["TitleNEW"].tolist()
    diseases_for_val = df_val["TitleNEW"].tolist()

    # 6. Initialize BERT tokenizer
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    vocab_size = tokenizer.vocab_size
    print(f"Tokenizer vocabulary size: {vocab_size}")

    # 7. Tokenize text
    train_encodings = process_terms(diseases_for_train, tokenizer, max_length=MAX_SEQ_LENGTH)
    val_encodings = process_terms(diseases_for_val, tokenizer, max_length=MAX_SEQ_LENGTH)

    # 8. Build model
    n_classes = len(label_encoder.classes_)
    model = build_model(vocab_size=vocab_size, max_length=MAX_SEQ_LENGTH, n_classes=n_classes)
    model.summary()

    # 9. Train model
    history = model.fit(
        train_encodings,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(val_encodings, y_val),
        verbose=1,
    )

    # 10. Optionally save model and label encoder mapping
    model.save("icd11_text_classifier.h5")

    forward_label_mapping = dict(zip(label_encoder.classes_, range(n_classes)))
    backward_label_mapping = dict(zip(range(n_classes), label_encoder.classes_))

    np.save("label_forward_mapping.npy", forward_label_mapping, allow_pickle=True)
    np.save("label_backward_mapping.npy", backward_label_mapping, allow_pickle=True)

    print("Training finished. Model and label mappings saved.")


#if __name__ == "__main__":
#    main()

## Step-by-step run

In [19]:
# 1. Load data
df = load_icd_data(DATA_PATH)
print(f"Loaded {len(df)} rows from {DATA_PATH}")
print(df.head())

Loaded 35562 rows from SimpleTabulation-ICD-11-MMS-ru.txt
                            Foundation URI  \
0  http://id.who.int/icd/entity/1435254666   
1   http://id.who.int/icd/entity/588616678   
2   http://id.who.int/icd/entity/135352227   
3   http://id.who.int/icd/entity/257068234   
4   http://id.who.int/icd/entity/416025325   

                                 Linearization URI  Code      BlockId  \
0  http://id.who.int/icd/release/11/mms/1435254666   NaN          NaN   
1   http://id.who.int/icd/release/11/mms/588616678   NaN  BlockL1-1A0   
2   http://id.who.int/icd/release/11/mms/135352227   NaN  BlockL2-1A0   
3   http://id.who.int/icd/release/11/mms/257068234  1A00          NaN   
4   http://id.who.int/icd/release/11/mms/416025325  1A01          NaN   

                                             TitleEN  \
0           Certain infectious or parasitic diseases   
1  - Gastroenteritis or colitis of infectious origin   
2                - - Bacterial intestinal infections   
3 

In [20]:
# 2. Add cleaned titles
df = add_clean_title_column(df, source_col=TITLE_COLUMN, target_col="TitleNEW")
print(df.head())

                            Foundation URI  \
0  http://id.who.int/icd/entity/1435254666   
1   http://id.who.int/icd/entity/588616678   
2   http://id.who.int/icd/entity/135352227   
3   http://id.who.int/icd/entity/257068234   
4   http://id.who.int/icd/entity/416025325   

                                 Linearization URI  Code      BlockId  \
0  http://id.who.int/icd/release/11/mms/1435254666   NaN          NaN   
1   http://id.who.int/icd/release/11/mms/588616678   NaN  BlockL1-1A0   
2   http://id.who.int/icd/release/11/mms/135352227   NaN  BlockL2-1A0   
3   http://id.who.int/icd/release/11/mms/257068234  1A00          NaN   
4   http://id.who.int/icd/release/11/mms/416025325  1A01          NaN   

                                             TitleEN  \
0           Certain infectious or parasitic diseases   
1  - Gastroenteritis or colitis of infectious origin   
2                - - Bacterial intestinal infections   
3                                      - - - Cholera   
4   

In [21]:
# 3. Compute NEW_ID according to ICD-11 hierarchy logic
df = new_group_id(df)
print(df.head())

                            Foundation URI  \
0   http://id.who.int/icd/entity/588616678   
1   http://id.who.int/icd/entity/135352227   
2   http://id.who.int/icd/entity/257068234   
3   http://id.who.int/icd/entity/416025325   
4  http://id.who.int/icd/entity/2080365623   

                                 Linearization URI  Code      BlockId  \
0   http://id.who.int/icd/release/11/mms/588616678   NaN  BlockL1-1A0   
1   http://id.who.int/icd/release/11/mms/135352227   NaN  BlockL2-1A0   
2   http://id.who.int/icd/release/11/mms/257068234  1A00          NaN   
3   http://id.who.int/icd/release/11/mms/416025325  1A01          NaN   
4  http://id.who.int/icd/release/11/mms/2080365623  1A02          NaN   

                                             TitleEN  \
0  - Gastroenteritis or colitis of infectious origin   
1                - - Bacterial intestinal infections   
2                                      - - - Cholera   
3     - - - Intestinal infection due to other Vibrio   
4   

In [24]:
# Build dictionary: disease title -> ICD‑11 code
df_for_dict = df.dropna(subset=["TitleNEW", CODE_COLUMN]).copy()

disease_to_icd = {
    row[CODE_COLUMN]: row["TitleNEW"]
    for _, row in df_for_dict[["TitleNEW", CODE_COLUMN]].iterrows()
}

print(list(disease_to_icd.items())[:3])

[('BlockL1-1A0', 'Инфекционный гастроэнтерит или колит без уточнения инфекционного агента'), ('BlockL1-1A6', 'Инфекции, передаваемые преимущественно половым путем, неуточненные'), ('BlockL1-1B1', 'Болезни, вызванные микобактериями, неуточненные')]


In [31]:
# 4. Prepare labels
df_labeled, y_encoded, label_encoder = prepare_labels(df, label_col=CODE_COLUMN)
print(f"Number of labeled examples: {len(df_labeled)}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Number of labeled examples: 18223
Number of classes: 291


In [32]:
# 5. Train / validation split
df_train, df_val, y_train, y_val = train_val_split(df_labeled, y_encoded)

diseases_for_train = df_train["TitleNEW"].tolist()
diseases_for_val = df_val["TitleNEW"].tolist()

In [33]:
# Selection of the max_length parameter
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

token_lens = []
for text in diseases_for_train:
    tokens = tokenizer.encode(text, max_length=512)
    token_lens.append(len(tokens))

from statistics import mean

print(mean(token_lens))
print(pd.Series(token_lens).quantile(0.95))

45.738939570615265
93.0


In [34]:
# 6. Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size
print(f"Tokenizer vocabulary size: {vocab_size}")

Tokenizer vocabulary size: 30522


In [35]:
# 7. Tokenize text
train_encodings = process_terms(diseases_for_train, tokenizer, max_length=MAX_SEQ_LENGTH)
val_encodings = process_terms(diseases_for_val, tokenizer, max_length=MAX_SEQ_LENGTH)

In [36]:
# 8. Build model
n_classes = len(label_encoder.classes_)
model = build_model(vocab_size=vocab_size, max_length=MAX_SEQ_LENGTH, n_classes=n_classes)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
# 9. Train model
history = model.fit(
    train_encodings,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(val_encodings, y_val),
    verbose=1,
)

Epoch 1/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 191s 398ms/step - accuracy: 0.0381 - loss: 5.1236 - val_accuracy: 0.1128 - val_loss: 4.6261
Epoch 2/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 176s 387ms/step - accuracy: 0.1486 - loss: 4.3844 - val_accuracy: 0.2132 - val_loss: 4.0038
Epoch 3/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 173s 379ms/step - accuracy: 0.2138 - loss: 3.8963 - val_accuracy: 0.2462 - val_loss: 3.7548
Epoch 4/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 212s 400ms/step - accuracy: 0.2560 - loss: 3.5956 - val_accuracy: 0.2876 - val_loss: 3.5262
Epoch 5/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 173s 379ms/step - accuracy: 0.2830 - loss: 3.4022 - val_accuracy: 0.3288 - val_loss: 3.3062
Epoch 6/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 175s 383ms/step - accuracy: 0.3190 - loss: 3.1592 - val_accuracy: 0.3406 - val_loss: 3.1966
Epoch 7/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 172s 376ms/step - accuracy: 0.3409 - loss: 3.0264 - val_accuracy: 0.3611 - val_loss: 3.0465
Epoch 8/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 171s 375ms/step - accuracy: 0.3716 -

In [38]:
# 10. Optionally save model and label encoder mapping
#model.save("icd11_text_classifier.h5")

forward_label_mapping = dict(zip(label_encoder.classes_, range(n_classes)))
backward_label_mapping = dict(zip(range(n_classes), label_encoder.classes_))

np.save("label_forward_mapping.npy", forward_label_mapping, allow_pickle=True)
np.save("label_backward_mapping.npy", backward_label_mapping, allow_pickle=True)

print("Training finished. Model and label mappings saved.")

Training finished. Model and label mappings saved.


## Test model

In [39]:
# remove some common words from the diagnosis
from nltk.stem import SnowballStemmer
import re

def remove_stemmed_words(text, words_to_remove):
    stemmer = SnowballStemmer('russian')
    stems = set(stemmer.stem(word) for word in words_to_remove)

    def replace(match):
        word = match.group(0)
        if stemmer.stem(word) in stems:
            return ''
        return word

    return re.sub(r'\b\w+\b', replace, text)

words_to_exclude = ['также', 'инициально', 'криз', 'рецидив']

In [40]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

In [ ]:
# data for testing
new_text = ["Наследственная гемолитическая анемия, неуточненная", "апластическая анемия неуточненная", "другие иммунодефицитные нарушения", "системная красная волчанка неуточненная", "Тромбоцитопения с 8 лет, у матери также тромбоцитопения", "новообразование левой доли щитовидной железы"]
true_value = ["BlockL1-3A0", "BlockL1-3A0", "BlockL1-4A0", "BlockL1-4A4", "BlockL1-3B1", "BlockL1-2F9"]

In [ ]:
y_true = [forward_label_mapping[val] for val in true_value]
print(y_true)

new_text = [value.lower() for value in new_text]
new_text = [re.sub(r'[^a-zA-Z0-9|^a-яА-Я|^ |^-]', '', value) for value in new_text]
new_text = [remove_stemmed_words(value, words_to_exclude) for value in new_text]

X_new_padded = process_terms(new_text,tokenizer)

prediction = model.predict(X_new_padded)
y_pred = list(np.argmax(prediction, axis=1))
print(y_pred)


accuracy = accuracy_score(y_true, y_pred)
print(accuracy)

precision = precision_score(y_true, y_pred, average='weighted')
print(precision)

recall = recall_score(y_true, y_pred, average='weighted')
print(recall)

[28, 28, 31, 32, 29, 27]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step
[np.int64(28), np.int64(28), np.int64(38), np.int64(32), np.int64(29), np.int64(22)]
0.6666666666666666
0.6666666666666666
0.6666666666666666


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
uploaded = files.upload()

Saving diagnosis_table_example.csv to diagnosis_table_example.csv


In [42]:
df_test = pd.read_csv("diagnosis_table_example.csv", sep="\t")
print(df_test)
test_samples = df_test["Диагноз"].to_list()

                                              Диагноз
0                      Наследственная тромбоцитопения
1             Тромбоцитопатия. Синдром Бернара Сулье.
2                        тромбоцитопения неуточненная
3   ОЛЛ,  В2, ранний комбинированный (костно-мозго...
4             аутоимунная гемолитическая анемия, криз
5   Наследственная гемолитическая анемия неуточненная
6              хроническая идиопатическая нейтропения
7   Злокачественное новообразование переднего сред...
8           Двухростковая цитопения неясной этиологии
9                Первичный иммунодефицит неуточненный
10  Злокачественное новообразование коротких косте...
11                              В-ОЛЛ, В2, инициально
12                                        ОМЛ, switch
13                              Т-ОЛЛ, Т3, инициально
14                                ОМЛ, М1, инициально
15                              В-ОЛЛ, В4, инициально
16  ХМПЗ (Истинная полицитемия, мутация гена JAK2V...
17                          

In [43]:
test_samples = [value.lower() for value in test_samples] #lower case
test_samples = [re.sub(r'[^a-zA-Z0-9|^a-яА-Я|^ |^-]', '', value) for value in test_samples] #remove special characters except spaces and hyphens
test_samples = [re.sub(r'\([^)]*\)', '', value) for value in test_samples] #delete the value in parentheses
test_samples = [remove_stemmed_words(value, words_to_exclude) for value in test_samples] #remove some common words
test_samples = [re.sub(r'\b\w*\d\w*\b', '', value) for value in test_samples] #remove words with numbers
test_samples = [re.sub(r'\s+', ' ', value) for value in test_samples]
print(test_samples)

['наследственная тромбоцитопения', 'тромбоцитопатия синдром бернара сулье', 'тромбоцитопения неуточненная', 'олл ранний комбинированный костно-мозговой нейрорецидив солидный компонент в левой орбите спо под ', 'аутоимунная гемолитическая анемия ', 'наследственная гемолитическая анемия неуточненная', 'хроническая идиопатическая нейтропения', 'злокачественное новообразование переднего средостения новообразование переднего средостения', 'двухростковая цитопения неясной этиологии', 'первичный иммунодефицит неуточненный', 'злокачественное новообразование коротких костей нижней конечности', 'в-олл ', 'омл switch', 'т-олл ', 'омл ', 'в-олл ', 'хмпз истинная полицитемия мутация гена ', 'омл ', 'хмпз эритроцитоз неуточненный', 'хмл']


In [44]:
#decoding of abbreviated names (synonyms)
dict_synonyms = {"омл": "острый миелоидный лейкоз", "т-олл": "т-клеточный острый лимфобластный лейкоз", "в-олл": "в-клеточный острый лимфобластный лейкоз", "пид":"первичный иммунодефицит", "олл": "острый лимфоидный лейкоз", "пидс":"первичные иммунодефицитные состояния"}

for key in dict_synonyms:
  test_samples = [re.sub(fr"\b{key}\b", dict_synonyms[key], value) for value in test_samples]

print(test_samples)

['наследственная тромбоцитопения', 'тромбоцитопатия синдром бернара сулье', 'тромбоцитопения неуточненная', 'острый лимфоидный лейкоз ранний комбинированный костно-мозговой нейрорецидив солидный компонент в левой орбите спо под ', 'аутоимунная гемолитическая анемия ', 'наследственная гемолитическая анемия неуточненная', 'хроническая идиопатическая нейтропения', 'злокачественное новообразование переднего средостения новообразование переднего средостения', 'двухростковая цитопения неясной этиологии', 'первичный иммунодефицит неуточненный', 'злокачественное новообразование коротких костей нижней конечности', 'в-клеточный острый лимфобластный лейкоз ', 'острый миелоидный лейкоз switch', 'т-клеточный острый лимфобластный лейкоз ', 'острый миелоидный лейкоз ', 'в-клеточный острый лимфобластный лейкоз ', 'хмпз истинная полицитемия мутация гена ', 'острый миелоидный лейкоз ', 'хмпз эритроцитоз неуточненный', 'хмл']


In [45]:
X_new_padded = process_terms(test_samples,tokenizer)

prediction = model.predict(X_new_padded)
y_pred = list(np.argmax(prediction, axis=1))
print(y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 521ms/step
[np.int64(51), np.int64(51), np.int64(51), np.int64(44), np.int64(50), np.int64(50), np.int64(89), np.int64(45), np.int64(120), np.int64(53), np.int64(45), np.int64(44), np.int64(44), np.int64(44), np.int64(44), np.int64(44), np.int64(215), np.int64(44), np.int64(184), np.int64(234)]


In [46]:
new_col_group_id = []
new_col_group_name = []
for label in y_pred:
  new_col_group_id.append(backward_label_mapping[label])
  new_col_group_name.append(disease_to_icd[backward_label_mapping[label]])

df_test["GROUP"] = new_col_group_id
df_test["DISEASE_GROUP"] = new_col_group_name
print(df_test)

                                              Диагноз        GROUP  \
0                      Наследственная тромбоцитопения  BlockL1-3B1   
1             Тромбоцитопатия. Синдром Бернара Сулье.  BlockL1-3B1   
2                        тромбоцитопения неуточненная  BlockL1-3B1   
3   ОЛЛ,  В2, ранний комбинированный (костно-мозго...  BlockL1-2A2   
4             аутоимунная гемолитическая анемия, криз  BlockL1-3A0   
5   Наследственная гемолитическая анемия неуточненная  BlockL1-3A0   
6              хроническая идиопатическая нейтропения  BlockL1-8A0   
7   Злокачественное новообразование переднего сред...  BlockL1-2B5   
8           Двухростковая цитопения неясной этиологии  BlockL1-AB3   
9                Первичный иммунодефицит неуточненный  BlockL1-4A0   
10  Злокачественное новообразование коротких косте...  BlockL1-2B5   
11                              В-ОЛЛ, В2, инициально  BlockL1-2A2   
12                                        ОМЛ, switch  BlockL1-2A2   
13                  